In [1]:
import os, numpy as np, pandas as pd, time
SH    = '/kaggle/input/datasets/abhiroopsarkar/knee-shards-all/shards'
LAB   = '/kaggle/input/datasets/abhiroopsarkar/knee-labels/report_labels.csv'
TRAIN = '/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv'
TMP   = '/kaggle/temp/xs'; os.makedirs(TMP, exist_ok=True)
LABELS = ['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA',
          'PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']

uids, masks = [], []
for ci in range(18):
    d = np.load(f'{SH}/shard_{ci:04d}.npz')
    if not os.path.exists(f'{TMP}/x_{ci:04d}.npy'):
        np.save(f'{TMP}/x_{ci:04d}.npy', d['x'])
    uids += list(d['uid']); masks.append(d['m'])
    print(ci, time.strftime('%H:%M:%S'), flush=True)
MASK = np.concatenate(masks).astype(np.float32)

lab = pd.read_csv(LAB).set_index('StudyInstanceUID').loc[uids]
trn = pd.read_csv(TRAIN).set_index('StudyInstanceUID').loc[uids]
Y    = lab[LABELS].values.astype(np.float32)
G    = trn[LABELS].values
gold = ~np.isnan(G).any(1)
print(len(uids), 'studies | gold:', gold.sum())

0 03:39:42
1 03:39:52
2 03:40:02
3 03:40:11
4 03:40:20
5 03:40:29
6 03:40:38
7 03:40:47
8 03:40:56
9 03:41:05
10 03:41:13
11 03:41:20
12 03:41:28
13 03:41:36
14 03:41:44
15 03:41:52
16 03:42:00
17 03:42:01
4407 studies | gold: 58


In [2]:
import warnings, timm, torch, torch.nn as nn, torch.nn.functional as F, time
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore', category=UserWarning)

BACKBONE, KU, EPOCHS, BS, LR = 'efficientnet_b3', 16, 7, 3, 3e-4
SMOKE = 0
dev = 'cuda'
torch.backends.cudnn.benchmark = True

class DS(Dataset):
    def __init__(s, idx, train): s.idx, s.train, s.mm = idx, train, {}
    def __len__(s): return len(s.idx)
    def __getitem__(s, j):
        i = s.idx[j]; ci = i // 256
        if ci not in s.mm: s.mm[ci] = np.load(f'{TMP}/x_{ci:04d}.npy', mmap_mode='r')
        sl = (np.sort(np.random.choice(24, KU, replace=False)) if s.train
              else np.linspace(0, 23, KU).round().astype(int))
        x = torch.from_numpy(np.array(s.mm[ci][i % 256][:, sl]))
        return x, torch.from_numpy(MASK[i]), torch.from_numpy(Y[i])

class Net(nn.Module):
    def __init__(s, pretrained=True):
        super().__init__()
        s.bb = timm.create_model(BACKBONE, pretrained=pretrained, num_classes=0, in_chans=1)
        D = s.bb.num_features
        s.att = nn.ModuleList([nn.Linear(D, 1) for _ in range(3)])
        s.drop = nn.Dropout(0.3)
        s.head = nn.Linear(3 * D + 3, 12)
    def forward(s, x, m):
        B, P, K, H, W = x.shape
        f = s.bb(x.reshape(B * P * K, 1, H, W)).float().reshape(B, P, K, -1)
        e = []
        for p in range(3):
            w = torch.softmax(s.att[p](f[:, p]).squeeze(-1), 1)
            e.append((w.unsqueeze(-1) * f[:, p]).sum(1))
        e = torch.stack(e, 1) * m.unsqueeze(-1)
        return s.head(s.drop(torch.cat([e.flatten(1), m], 1)))

def prep(x): return (x.float() / 255 - 0.2) / 0.25
def aug(x):
    B = x.shape[0]
    x = x * (1 + 0.15 * torch.randn(B, 1, 1, 1, 1, device=x.device)) + 0.15 * torch.randn(B, 1, 1, 1, 1, device=x.device)
    dy, dx = np.random.randint(-10, 11, 2)
    return torch.roll(x, (int(dy), int(dx)), (3, 4))

def macro_auc(P, T):
    T = T >= 0.5
    a = [roc_auc_score(T[:, k], P[:, k]) for k in range(12) if 0 < T[:, k].sum() < len(T)]
    return float(np.mean(a))

@torch.no_grad()
def predict(model, idx):
    model.eval(); out = []
    for x, m, _ in DataLoader(DS(idx, False), batch_size=6, num_workers=4):
        with torch.autocast('cuda', dtype=torch.float16):
            out.append(model(prep(x.to(dev)), m.to(dev)).float().sigmoid().cpu())
    return torch.cat(out).numpy()

rng = np.random.RandomState(0)
nong = np.where(~gold)[0]; rng.shuffle(nong)
nv = int(0.15 * len(nong)); val_idx, tr_idx = nong[:nv], nong[nv:]
gold_idx = np.where(gold)[0]
if SMOKE: tr_idx, val_idx = tr_idx[:SMOKE], val_idx[:SMOKE // 4]
print('train', len(tr_idx), '| val', len(val_idx), '| gold', len(gold_idx), flush=True)

model = Net().to(dev).to(memory_format=torch.channels_last)
dl = DataLoader(DS(tr_idx, True), batch_size=BS, shuffle=True, num_workers=4, drop_last=True, pin_memory=True)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(dl), pct_start=0.1)
scaler = torch.amp.GradScaler('cuda')
best = 0
for ep in range(EPOCHS):
    model.train(); t0 = time.time(); tl = 0
    for it, (x, m, y) in enumerate(dl):
        x, m, y = aug(prep(x.to(dev))), m.to(dev), y.to(dev)
        with torch.autocast('cuda', dtype=torch.float16):
            logit = model(x, m)
        loss = F.binary_cross_entropy_with_logits(logit.float(), y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tl += loss.item()
        if it % 400 == 0: print(f'ep{ep} it{it}/{len(dl)} loss {tl/(it+1):.4f} {time.time()-t0:.0f}s', flush=True)
    Pv, Pg = predict(model, val_idx), predict(model, gold_idx)
    pv, pg = macro_auc(Pv, Y[val_idx]), macro_auc(Pg, G[gold_idx])
    Tg = G[gold_idx] >= 0.5
    mm = roc_auc_score(Tg[:, 2], Pg[:, 2]); lm = roc_auc_score(Tg[:, 3], Pg[:, 3])
    print(f'=== ep{ep} loss {tl/len(dl):.4f} | val {pv:.3f} | GOLD {pg:.3f} | MedMen {mm:.3f} | LatMen {lm:.3f} | {time.time()-t0:.0f}s', flush=True)
    if pv > best: best = pv; torch.save(model.state_dict(), '/kaggle/working/model_b3.pt')

train 3697 | val 652 | gold 58


model.safetensors: reconstructing file:   0%|          |  0.00B / 49.3MB            

model.safetensors: downloading bytes:           |  0.00B            

ep0 it0/1232 loss 0.6970 78s
ep0 it400/1232 loss 0.5424 246s
ep0 it800/1232 loss 0.5255 415s
ep0 it1200/1232 loss 0.5122 584s
=== ep0 loss 0.5105 | val 0.787 | GOLD 0.808 | MedMen 0.683 | LatMen 0.759 | 680s
ep1 it0/1232 loss 0.4735 1s
ep1 it400/1232 loss 0.4568 170s
ep1 it800/1232 loss 0.4456 339s
ep1 it1200/1232 loss 0.4417 508s
=== ep1 loss 0.4408 | val 0.837 | GOLD 0.835 | MedMen 0.688 | LatMen 0.729 | 551s
ep2 it0/1232 loss 0.3309 1s
ep2 it400/1232 loss 0.4026 170s
ep2 it800/1232 loss 0.4018 338s
ep2 it1200/1232 loss 0.4023 507s
=== ep2 loss 0.4024 | val 0.851 | GOLD 0.859 | MedMen 0.745 | LatMen 0.766 | 550s
ep3 it0/1232 loss 0.3581 1s
ep3 it400/1232 loss 0.3658 170s
ep3 it800/1232 loss 0.3684 339s
ep3 it1200/1232 loss 0.3686 508s
=== ep3 loss 0.3688 | val 0.862 | GOLD 0.868 | MedMen 0.809 | LatMen 0.750 | 550s
ep4 it0/1232 loss 0.2461 1s
ep4 it400/1232 loss 0.3350 170s
ep4 it800/1232 loss 0.3327 338s
ep4 it1200/1232 loss 0.3319 507s
=== ep4 loss 0.3318 | val 0.864 | GOLD 0.881 |